In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [10]:
fake_df = pd.read_csv("dataset/Fake.csv")
true_df = pd.read_csv("dataset/True.csv")

In [ ]:
# Load the dataset
iris = load_iris()
X, y = iris.data, iris.target
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Initialize the XGBClassifier
model = xgb.XGBClassifier()
# Train the model
model.fit(X_train, y_train)
# Make predictions
y_pred = model.predict(X_test)
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 100.00%


In [12]:
fake_df['category'] = 'fake'
true_df['category'] = 'true'


In [13]:
df = pd.concat([true_df, fake_df], ignore_index=True)

In [19]:
df['label'] = df['category'].map({'fake': 0, 'true': 1})

In [21]:
df.tail()

,title,text,subject,date,category,label
44893,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",fake,0
44894,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",fake,0
44895,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",fake,0
44896,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",fake,0
44897,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",fake,0


In [23]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = [
    "The weather is lovely today. It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1835.68it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(2, 384)
tensor([[1.0000, 0.0918],
        [0.0918, 1.0000]])


In [26]:
df['text'].describe()

count     44898
unique    38646
top            
freq        627
Name: text, dtype: object

In [27]:
df['sentence_emb'] = df['text'].apply(lambda x: model.encode(x))

In [28]:
df.head()

,title,text,subject,date,category,label,sentence_emb
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",true,1,"[0.030698761, -0.013403436, 0.007463461, 0.037..."
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",true,1,"[-0.032901783, -0.0034171245, 0.039577432, -0...."
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",true,1,"[-0.029781317, -0.040301643, 0.0386762, 0.0211..."
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",true,1,"[-0.03850345, -0.0077626463, 0.009181647, -0.0..."
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",true,1,"[0.021806808, -0.019007133, 0.042010445, 0.036..."


In [33]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb

X = np.stack(df['sentence_emb'].values)  # shape: (n_samples, embedding_dim)
y = df['label'].values

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model = xgb.XGBClassifier()
model.fit(X_train, y_train)

# Predict & evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 97.15%


In [34]:
import internetarchive

search = internetarchive.search_items(
    'collection:opensource_texts AND format:Text'
)

for result in search:
    identifier = result['identifier']
    internetarchive.download(identifier, formats=['Text'], destdir='downloads')

In [35]:
for result in search:
    print(result)

In [ ]:
import requests, os

SEARCH_API = "https://archive.org/advancedsearch.php"
META_API = "https://archive.org/metadata/"
DL_BASE = "https://archive.org/download/"

def search_americana_cookery(rows=500):
    page = 1
    while True:
        params = {
            "q": "collection:americana AND (subject:cookery OR subject:cooking) AND format:Text",
            "fl[]": "identifier",
            "rows": rows,
            "page": page,
            "output": "json"
        }
        resp = requests.get(SEARCH_API, params=params).json()
        docs = resp["response"]["docs"]
        if not docs:
            break
        for d in docs:
            yield d["identifier"]
        if len(docs) < rows:
            break
        page += 1

def download_txt_files(identifier, save_dir="americana_txts"):
    os.makedirs(save_dir, exist_ok=True)
    meta = requests.get(META_API + identifier).json()
    for f in meta.get("files", []):
        if f.get("name", "").endswith(".txt"):
            url = f"{DL_BASE}{identifier}/{f['name']}"
            r = requests.get(url)
            with open(os.path.join(save_dir, f["name"]), "wb") as out:
                out.write(r.content)
            print("Saved:", f["name"])

for item_id in search_americana_cookery():
    download_txt_files(item_id)

Downloaded: cbk_zamwiki-20220910-md5sums.txt
Downloaded: maxrevid.txt
Downloaded: status.txt
Downloaded: cbk_zamwiki-20140802-md5sums.txt
Downloaded: maxrevid.txt
Downloaded: status.txt
Downloaded: cbk_zamwiki-20150810-md5sums.txt
Downloaded: maxrevid.txt
Downloaded: status.txt


KeyboardInterrupt: 